In [ ]:
# submit.py
import numpy as np

def my_decode(w):
    """
    Input:
      w: 1D numpy array of length 1089 (=(k+1)^2) representing the XOR arbiter PUF linear model.
    Output:
      eight numpy arrays (a,b,c,d,p,q,r,s), each shape (32,), containing non-negative delays.
    Approach:
      - Reshape w to (33,33) matrix W (row-major / C order).
      - SVD rank-1 decompose W -> u_est, v_est.
      - For each arbiter model m (u_est, v_est) pick alpha/beta decomposition:
          alpha_0 = m[0], alpha_i=0 for i>=1, beta_i = m[i+1]
      - For each stage compute deltas and map them to non-negative pairs:
          (p,q) from delta1 = alpha+beta, (r,s) from delta2 = alpha-beta.
    """
    k = 32
    expected_len = (k+1)**2
    w = np.asarray(w, dtype=np.float64).ravel()
    if w.size != expected_len:
        raise ValueError(f"Expected input vector of length {expected_len}, got {w.size}")

    # 1. De-Kronecker via SVD
    W = w.reshape((k+1, k+1))   # row-major (C order), consistent with numpy.kron
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    # Numerical guard: if S[0] negative (shouldn't be) or zero, handle gracefully.
    root = np.sqrt(S[0]) if S[0] > 0 else 0.0
    u_est = U[:, 0] * root
    v_est = Vt[0, :] * root

    # Fix sign: ensure last coordinate of v_est is non-negative for deterministic output
    if v_est[-1] < 0:
        u_est = -u_est
        v_est = -v_est

    def model_to_delays(m):
        """Given arbiter model m (length k+1), produce four length-k non-negative delay vectors."""
        # canonical alpha/beta choice
        alpha = np.zeros(k, dtype=np.float64)
        alpha[0] = m[0]
        beta = m[1:].copy()  # length k

        p = np.zeros(k, dtype=np.float64)
        q = np.zeros(k, dtype=np.float64)
        r = np.zeros(k, dtype=np.float64)
        s = np.zeros(k, dtype=np.float64)

        # For each stage compute difference and choose non-negative representative
        for i in range(k):
            delta1 = alpha[i] + beta[i]   # corresponds to x_i - y_i
            delta2 = alpha[i] - beta[i]   # corresponds to u_i - v_i

            if delta1 >= 0:
                p[i] = delta1
                q[i] = 0.0
            else:
                p[i] = 0.0
                q[i] = -delta1

            if delta2 >= 0:
                r[i] = delta2
                s[i] = 0.0
            else:
                r[i] = 0.0
                s[i] = -delta2

        # Clamp any tiny negative numerical noise to zero
        for arr in (p, q, r, s):
            arr[arr < 0] = 0.0

        return p, q, r, s

    # Convert u_est and v_est to delays
    a, b, c, d = model_to_delays(u_est)
    p, q, r, s = model_to_delays(v_est)

    # Final safety clamp (shouldn't be necessary)
    for arr in (a, b, c, d, p, q, r, s):
        arr[arr < 0] = 0.0

    return a, b, c, d, p, q, r, s


# Optional test harness (not required for submission) -----------------------
if __name__ == "__main__":
    # quick self-test: create a random consistent model and decode it
    k = 32
    # create random nonnegative delays for two pufs, then produce w
    rng = np.random.default_rng(12345)
    def random_delays():
        return [rng.random(k) for _ in range(4)]
    a0, b0, c0, d0 = random_delays()
    p0, q0, r0, s0 = random_delays()
    # helper to build model vector from delays
    def delays_to_model(x, y, u, v):
        alpha = (x - y + u - v)/2
        beta =  (x - y - u + v)/2
        m = np.zeros(k+1)
        m[0] = alpha[0]
        m[1:-1] = alpha[1:] + beta[:-1]
        m[-1] = beta[-1]
        return m
    u = delays_to_model(a0, b0, c0, d0)
    v = delays_to_model(p0, q0, r0, s0)
    w = np.kron(u, v)
    out = my_decode(w)
    # Rebuild models from recovered delays to check error
    def rebuild_model_from_delays(x,y,u,v):
        alpha = (x - y + u - v)/2
        beta  = (x - y - u + v)/2
        m = np.zeros(k+1)
        m[0] = alpha[0]
        m[1:-1] = alpha[1:] + beta[:-1]
        m[-1] = beta[-1]
        return m
    a1,b1,c1,d1,p1,q1,r1,s1 = out
    u_rec = rebuild_model_from_delays(a1,b1,c1,d1)
    v_rec = rebuild_model_from_delays(p1,q1,r1,s1)
    w_hat = np.kron(u_rec, v_rec)
    err = np.linalg.norm(w - w_hat)
    print("Self-test reconstruction error:", err)


In [ ]:
# Running the robust decoding on the uploaded file and saving outputs.
import numpy as np, os, re, textwrap
from pathlib import Path

K = 32
INPUT_FILE = "/content/public_mod.txt"
OUTPUT_DIR = "/content"

_float_re = re.compile(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?')

def load_models_from_file(path: str, k: int = K):
    expected_len = (k+1)**2
    with open(path, "r") as f:
        txt = f.read()
    tokens = _float_re.findall(txt)
    vals = [float(t) for t in tokens]
    if len(vals) % expected_len != 0:
        raise ValueError(f"After extracting floats found {len(vals)} numbers; expected multiple of {expected_len}.")
    n_models = len(vals) // expected_len
    models = [np.array(vals[i*expected_len:(i+1)*expected_len], dtype=np.float64) for i in range(n_models)]
    return models

def dekronecker(w: np.ndarray, k: int = K):
    W = w.reshape((k+1, k+1))
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    root = np.sqrt(S[0]) if S[0] > 0 else 0.0
    u_est = U[:, 0] * root
    v_est = Vt[0, :] * root
    if v_est[-1] < 0:
        u_est = -u_est
        v_est = -v_est
    return u_est, v_est

def model_to_alpha_beta(m: np.ndarray, k: int = K):
    alpha = np.zeros(k, dtype=np.float64)
    alpha[0] = m[0]
    beta = m[1:].copy()
    return alpha, beta

def alpha_beta_to_delays(alpha: np.ndarray, beta: np.ndarray, k: int = K):
    p = np.zeros(k, dtype=np.float64)
    q = np.zeros(k, dtype=np.float64)
    r = np.zeros(k, dtype=np.float64)
    s = np.zeros(k, dtype=np.float64)
    for i in range(k):
        delta1 = alpha[i] + beta[i]
        delta2 = alpha[i] - beta[i]
        if delta1 >= 0:
            p[i] = delta1; q[i] = 0.0
        else:
            p[i] = 0.0; q[i] = -delta1
        if delta2 >= 0:
            r[i] = delta2; s[i] = 0.0
        else:
            r[i] = 0.0; s[i] = -delta2
    for arr in (p,q,r,s):
        arr[arr < 0] = 0.0
    return p, q, r, s

def my_decode(w: np.ndarray, k: int = K):
    w = np.asarray(w, dtype=np.float64).ravel()
    if w.size != (k+1)**2:
        raise ValueError(f"Expected vector length {(k+1)**2}, got {w.size}")
    u_est, v_est = dekronecker(w, k)
    alpha_u, beta_u = model_to_alpha_beta(u_est, k)
    a,b,c,d = alpha_beta_to_delays(alpha_u, beta_u, k)
    alpha_v, beta_v = model_to_alpha_beta(v_est, k)
    p,q,r,s = alpha_beta_to_delays(alpha_v, beta_v, k)
    for arr in (a,b,c,d,p,q,r,s):
        arr[arr < 0] = 0.0
    return a,b,c,d,p,q,r,s

def rebuild_model_from_delays(a,b,c,d,k=K):
    alpha = (a - b + c - d) / 2.0
    beta  = (a - b - c + d) / 2.0
    m = np.zeros(k+1, dtype=np.float64)
    m[0] = alpha[0]
    if k > 1:
        m[1:-1] = alpha[1:] + beta[:-1]
    m[-1] = beta[-1]
    return m

def reconstruct_w_from_delays(a,b,c,d,p,q,r,s,k=K):
    u = rebuild_model_from_delays(a,b,c,d,k)
    v = rebuild_model_from_delays(p,q,r,s,k)
    return np.kron(u, v)

# Load models
models = load_models_from_file(INPUT_FILE, K)
n = len(models)
print(f"Loaded {n} models.")

# Decode all models and save results
os.makedirs(OUTPUT_DIR, exist_ok=True)
all_delays = []
errors = []
for idx, w in enumerate(models):
    a,b,c,d,p,q,r,s = my_decode(w, K)
    all_del = np.concatenate([a,b,c,d,p,q,r,s])
    all_delays.append(all_del)
    w_hat = reconstruct_w_from_delays(a,b,c,d,p,q,r,s,K)
    err = np.linalg.norm(w - w_hat)
    errors.append(err)
    model_dir = Path(OUTPUT_DIR)/f"model_{idx}"
    model_dir.mkdir(parents=True, exist_ok=True)
    np.savez(model_dir / f"delays_model_{idx}.npz", a=a,b=b,c=c,d=d,p=p,q=q,r=r,s=s)
    np.savetxt(model_dir / f"delays_model_{idx}_flat.csv", all_del[None,:], delimiter=",", fmt="%.12g")
    print(f"Model {idx}: L2 error = {err:.3e}  -> saved to {model_dir}")

# Save all together
np.savez(Path(OUTPUT_DIR)/"all_recovered_delays.npz", delays=np.vstack(all_delays), errors=np.array(errors))
print("\nSaved all outputs to:", OUTPUT_DIR)
print("Per-model errors:", errors)


In [ ]:
# ============================
# Final Robust XOR-PUF Decoder
# ============================

import numpy as np
import re, os, time
from pathlib import Path

K = 32
INPUT_FILE = "/content/public_mod.txt"   # <-- your file
OUTPUT_DIR = "/content/puf_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Float Regex Loader (Robust)
# -----------------------------
_FLOAT_RE = re.compile(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?')

def load_models_from_file(path: str, k: int = K):
    """
    Loads all numeric tokens from a text file and reshapes into models of length (k+1)^2 = 1089.
    """
    expected = (k+1)**2
    with open(path, "r", errors="ignore") as f:
        txt = f.read()

    tokens = _FLOAT_RE.findall(txt)
    vals = np.array([float(t) for t in tokens], dtype=np.float64)

    if len(vals) % expected != 0:
        raise ValueError(
            f"Extracted {len(vals)} numeric tokens, expected a multiple of {expected}.\n"
            f"Preview first 30 tokens: {tokens[:30]}"
        )

    n = len(vals) // expected
    models = [vals[i*expected:(i+1)*expected].copy() for i in range(n)]
    print(f"Loaded {n} model(s).")
    return models

# -----------------------------
# Core SVD-based Decoder
# -----------------------------
def my_decode(w: np.ndarray):
    """
    Returns eight 32-length non-negative delay arrays (a,b,c,d,p,q,r,s).
    """
    k = K
    w = np.asarray(w, dtype=np.float64).ravel()
    if w.size != (k+1)**2:
        raise ValueError("Input model must have length 1089 = (32+1)^2")

    # reshape model to rank-1 matrix
    W = w.reshape((k+1, k+1))

    # SVD rank-1 factorization
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    root = np.sqrt(S[0])
    u_est = U[:, 0] * root
    v_est = Vt[0, :] * root

    # fix sign ambiguity
    if v_est[-1] < 0:
        u_est = -u_est
        v_est = -v_est

    # Extract α, β for Arbiter 1
    alpha_u = np.zeros(k)
    alpha_u[0] = u_est[0]
    beta_u = u_est[1:].copy()

    # Extract α, β for Arbiter 2
    alpha_v = np.zeros(k)
    alpha_v[0] = v_est[0]
    beta_v = v_est[1:].copy()

    # Convert α, β → delay differences
    delta1_u = alpha_u + beta_u
    delta2_u = alpha_u - beta_u

    delta1_v = alpha_v + beta_v
    delta2_v = alpha_v - beta_v

    # Construct non-negative delays
    a = np.maximum(delta1_u, 0)
    b = np.maximum(-delta1_u, 0)
    c = np.maximum(delta2_u, 0)
    d = np.maximum(-delta2_u, 0)

    p = np.maximum(delta1_v, 0)
    q = np.maximum(-delta1_v, 0)
    r = np.maximum(delta2_v, 0)
    s = np.maximum(-delta2_v, 0)

    return a, b, c, d, p, q, r, s

# -----------------------------
# Rebuild model
# -----------------------------
def rebuild_model_from_delays(a,b,c,d,k=K):
    alpha = (a - b + c - d) / 2.0
    beta  = (a - b - c + d) / 2.0

    m = np.zeros(k+1)
    m[0] = alpha[0]
    if k > 1:
        m[1:-1] = alpha[1:] + beta[:-1]
    m[-1] = beta[-1]
    return m


def reconstruct_w_from_delays(a,b,c,d,p,q,r,s,k=K):
    u = rebuild_model_from_delays(a,b,c,d,k)
    v = rebuild_model_from_delays(p,q,r,s,k)
    return np.kron(u, v)

# -----------------------------
# Run Decoder on All Models
# -----------------------------
models = load_models_from_file(INPUT_FILE)
all_delays = []
errors = []

print("\nDecoding all models...\n")

for idx, w in enumerate(models):
    a,b,c,d,p,q,r,s = my_decode(w)
    all_del = np.concatenate([a,b,c,d,p,q,r,s])

    # reconstruct to check error
    w_hat = reconstruct_w_from_delays(a,b,c,d,p,q,r,s)
    err = np.linalg.norm(w - w_hat)
    errors.append(err)
    all_delays.append(all_del)

    # save results
    model_dir = Path(OUTPUT_DIR) / f"model_{idx}"
    model_dir.mkdir(exist_ok=True, parents=True)

    np.savez(model_dir/"delays.npz", a=a,b=b,c=c,d=d,p=p,q=q,r=r,s=s)
    np.savetxt(model_dir/"delays_flat.csv", all_del.reshape(1,-1), delimiter=",", fmt="%.12g")

    print(f"Model {idx}: Reconstruction error = {err:.3e}")

# save all models
np.savez(Path(OUTPUT_DIR)/"all_recovered_delays.npz",
         delays=np.vstack(all_delays),
         errors=np.array(errors))

print("\nAll results saved to:", OUTPUT_DIR)
print("Final per-model errors:", errors)


In [ ]:
#!/usr/bin/env python3
"""
submit.py

Robust submission file for Problem 1.2.

- Implements my_decode(w) which takes a single 1089-dimensional model vector and
  returns eight 32-dimensional non-negative delay vectors: (a,b,c,d,p,q,r,s).
- Provides a safe main() / evaluation harness that ignores Jupyter/Colab kernel args
  and uses the uploaded model file by default (/mnt/data/public_mod.txt).
"""

import sys
import os
import re
import time
from pathlib import Path
import numpy as np

# ---------- configuration ----------
K = 32
DEFAULT_INPUT = "/content/secret_mod.txt"   # fallback / sample file (uploaded)
OUTPUT_DIR = "/mnt/data/puf_recovery_output"
REPEATS = 5
# -----------------------------------

# float regex (robust)
_FLOAT_RE = re.compile(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?')

# ---------------- loader ----------------
def load_models_from_file(path: str, k: int = K, verbose: bool = True):
    expected = (k + 1) ** 2
    with open(path, "r", errors="ignore") as f:
        txt = f.read()
    tokens = _FLOAT_RE.findall(txt)
    if verbose:
        print(f"[loader] extracted {len(tokens)} numeric tokens from '{path}'")
    if len(tokens) == 0:
        raise ValueError(f"No numeric tokens found in file: {path}")
    vals = np.array([float(t) for t in tokens], dtype=np.float64)
    if vals.size % expected != 0:
        preview = " ".join(tokens[:min(60, len(tokens))])
        raise ValueError(
            f"After extracting floats found {vals.size} numbers; expected a multiple of {expected}.\n"
            f"Preview (first tokens): {preview}\n"
            "Please supply the correct models file (1089 floats per model)."
        )
    n = vals.size // expected
    models = [vals[i*expected:(i+1)*expected].copy() for i in range(n)]
    if verbose:
        print(f"[loader] loaded {n} model(s), each length {expected}.")
    return models

# ---------------- decoder ----------------
def my_decode(w: np.ndarray):
    """
    Input:
        w: numpy array length 1089 ((K+1)^2)
    Output:
        tuple of eight numpy arrays (a,b,c,d,p,q,r,s), each shape (K,)
        All outputs are non-negative (zeros allowed).
    """
    expect = (K+1)**2
    w = np.asarray(w, dtype=np.float64).ravel()
    if w.size != expect:
        raise ValueError(f"my_decode expects length {expect}, got {w.size}")

    # reshape and top-rank SVD to de-Kronecker
    W = w.reshape((K+1, K+1))
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    s0 = S[0] if S.size > 0 else 0.0
    root = np.sqrt(s0) if s0 > 0.0 else 0.0
    u_est = U[:, 0] * root
    v_est = Vt[0, :] * root

    # deterministic sign choice
    if v_est[-1] < 0.0:
        u_est = -u_est
        v_est = -v_est

    # canonical decomposition to alpha/beta
    alpha_u = np.zeros(K, dtype=np.float64)
    alpha_u[0] = u_est[0]
    beta_u = u_est[1:].copy()

    alpha_v = np.zeros(K, dtype=np.float64)
    alpha_v[0] = v_est[0]
    beta_v = v_est[1:].copy()

    # compute deltas and map to non-negative delays
    delta1_u = alpha_u + beta_u
    delta2_u = alpha_u - beta_u
    a = np.where(delta1_u >= 0.0, delta1_u, 0.0)
    b = np.where(delta1_u <  0.0, -delta1_u, 0.0)
    c = np.where(delta2_u >= 0.0, delta2_u, 0.0)
    d = np.where(delta2_u <  0.0, -delta2_u, 0.0)

    delta1_v = alpha_v + beta_v
    delta2_v = alpha_v - beta_v
    p = np.where(delta1_v >= 0.0, delta1_v, 0.0)
    q = np.where(delta1_v <  0.0, -delta1_v, 0.0)
    r = np.where(delta2_v >= 0.0, delta2_v, 0.0)
    s = np.where(delta2_v <  0.0, -delta2_v, 0.0)

    # clamp tiny negative numerical noise
    for arr in (a,b,c,d,p,q,r,s):
        arr[arr < 0.0] = 0.0

    return a, b, c, d, p, q, r, s

# ---------------- reconstruction ----------------
def rebuild_model_from_delays(a, b, c, d):
    alpha = (a - b + c - d) / 2.0
    beta  = (a - b - c + d) / 2.0
    m = np.zeros(K+1, dtype=np.float64)
    m[0] = alpha[0]
    if K > 1:
        m[1:-1] = alpha[1:] + beta[:-1]
    m[-1] = beta[-1]
    return m

def reconstruct_w_from_delays(a,b,c,d,p,q,r,s):
    u = rebuild_model_from_delays(a,b,c,d)
    v = rebuild_model_from_delays(p,q,r,s)
    return np.kron(u, v)

# ---------------- evaluation harness ----------------
def evaluate_models(models, repeats=REPEATS, out_dir=OUTPUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    per_model_summary = []
    all_delays = []
    for i, w in enumerate(models):
        # warm-up
        _ = my_decode(w)
        times = []
        errors = []
        rep_delays = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            a,b,c,d,p,q,r,s = my_decode(w)
            t1 = time.perf_counter()
            elapsed = t1 - t0
            # clamp negatives (mimic grader)
            for arr in (a,b,c,d,p,q,r,s):
                arr[arr < 0.0] = 0.0
            w_hat = reconstruct_w_from_delays(a,b,c,d,p,q,r,s)
            err = float(np.linalg.norm(w - w_hat))
            times.append(elapsed)
            errors.append(err)
            if rep_delays is None:
                rep_delays = np.concatenate([a,b,c,d,p,q,r,s])
        mean_time = float(np.mean(times))
        mean_err = float(np.mean(errors))
        per_model_summary.append({
            "model_index": int(i),
            "mean_time_s": mean_time,
            "mean_L2_error": mean_err,
            "times": times,
            "errors": errors
        })
        all_delays.append(rep_delays)
        # save per-model delays
        md = Path(out_dir) / f"model_{i}"
        md.mkdir(parents=True, exist_ok=True)
        a0 = rep_delays[0:K]; b0 = rep_delays[K:2*K]; c0 = rep_delays[2*K:3*K]; d0 = rep_delays[3*K:4*K]
        p0 = rep_delays[4*K:5*K]; q0 = rep_delays[5*K:6*K]; r0 = rep_delays[6*K:7*K]; s0 = rep_delays[7*K:8*K]
        np.savez(md/"delays.npz", a=a0, b=b0, c=c0, d=d0, p=p0, q=q0, r=r0, s=s0)
        np.savetxt(md/"delays_flat.csv", rep_delays[None,:], delimiter=",", fmt="%.12g")
        print(f"[Model {i}] mean_time = {mean_time:.6e} s, mean_L2_error = {mean_err:.6e}")
    # overall summary
    mean_time_all = float(np.mean([rec["mean_time_s"] for rec in per_model_summary]))
    mean_err_all = float(np.mean([rec["mean_L2_error"] for rec in per_model_summary]))
    np.savez(Path(out_dir)/"results_summary.npz",
             per_model_summary=per_model_summary,
             mean_time_all=mean_time_all,
             mean_err_all=mean_err_all,
             all_delays=np.vstack(all_delays) if all_delays else np.empty((0, 8*K)))
    print("\nOverall mean time (s):", mean_time_all)
    print("Overall mean L2 error :", mean_err_all)
    print("Results and per-model delays saved at:", out_dir)
    return per_model_summary, mean_time_all, mean_err_all

# ---------------- main ----------------
def main(argv):
    # Robust argument handling:
    # - ignore Jupyter/Colab auto args like '-f' or kernel.json
    # - if user provided an explicit path that exists and contains valid tokens, use it
    # - otherwise fall back to DEFAULT_INPUT
    args = [a for a in argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    input_path = None
    if len(args) >= 1:
        cand = args[0]
        if os.path.exists(cand):
            try:
                # quick check: does it contain at least one full model?
                _ = load_models_from_file(cand, k=K, verbose=False)
                input_path = cand
            except Exception as e:
                print(f"[main] Ignoring provided file '{cand}': {e}")
    if input_path is None:
        # fallback to the known uploaded sample file path
        input_path = DEFAULT_INPUT
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"No valid input found. Tried CLI args and fallback '{DEFAULT_INPUT}'")

    print("[main] Using input file:", input_path)
    models = load_models_from_file(input_path, k=K, verbose=True)
    evaluate_models(models, repeats=REPEATS, out_dir=OUTPUT_DIR)

if __name__ == "__main__":
    main(sys.argv)


[main] Using input file: /content/secret_mod.txt
[loader] extracted 10890 numeric tokens from '/content/secret_mod.txt'
[loader] loaded 10 model(s), each length 1089.
[Model 0] mean_time = 1.751303e-03 s, mean_L2_error = 2.526150e-16
[Model 1] mean_time = 2.665692e-04 s, mean_L2_error = 3.660798e-16
[Model 2] mean_time = 2.424658e-04 s, mean_L2_error = 3.967353e-16
[Model 3] mean_time = 2.434022e-04 s, mean_L2_error = 4.468876e-16
[Model 4] mean_time = 2.383858e-04 s, mean_L2_error = 4.817962e-16
[Model 5] mean_time = 2.390796e-04 s, mean_L2_error = 1.972229e-16
[Model 6] mean_time = 2.712018e-04 s, mean_L2_error = 7.646264e-16
[Model 7] mean_time = 2.240386e-04 s, mean_L2_error = 3.465333e-16
[Model 8] mean_time = 2.312420e-04 s, mean_L2_error = 3.833901e-16
[Model 9] mean_time = 2.246106e-04 s, mean_L2_error = 6.578025e-16

Overall mean time (s): 0.00039322983999909414
Overall mean L2 error : 4.2936891338804026e-16
Results and per-model delays saved at: /mnt/data/puf_recovery_output


In [ ]:
#!/usr/bin/env python3
"""
submit.py

Robust submission file for Problem 1.2.

- Implements my_decode(w) which takes a single 1089-dimensional model vector and
  returns eight 32-dimensional non-negative delay vectors: (a,b,c,d,p,q,r,s).
- Provides a safe main() / evaluation harness that ignores Jupyter/Colab kernel args
  and uses the uploaded model file by default (/content/secret_mod.txt).
"""

import sys
import os
import re
import time
from pathlib import Path
import numpy as np

# ---------- configuration ----------
K = 32
DEFAULT_INPUT = "//content/secret_mod.txt"   # fallback / sample file (uploaded)
OUTPUT_DIR = "/mnt/data/puf_recovery_output"
REPEATS = 5
# -----------------------------------

# float regex (robust)
_FLOAT_RE = re.compile(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?')

# ---------------- loader ----------------
def load_models_from_file(path: str, k: int = K, verbose: bool = True):
    expected = (k + 1) ** 2
    with open(path, "r", errors="ignore") as f:
        txt = f.read()
    tokens = _FLOAT_RE.findall(txt)
    if verbose:
        print(f"[loader] extracted {len(tokens)} numeric tokens from '{path}'")
    if len(tokens) == 0:
        raise ValueError(f"No numeric tokens found in file: {path}")
    vals = np.array([float(t) for t in tokens], dtype=np.float64)
    if vals.size % expected != 0:
        preview = " ".join(tokens[:min(60, len(tokens))])
        raise ValueError(
            f"After extracting floats found {vals.size} numbers; expected a multiple of {expected}.\n"
            f"Preview (first tokens): {preview}\n"
            "Please supply the correct models file (1089 floats per model)."
        )
    n = vals.size // expected
    models = [vals[i*expected:(i+1)*expected].copy() for i in range(n)]
    if verbose:
        print(f"[loader] loaded {n} model(s), each length {expected}.")
    return models


# ---------- single-PUF linear system A (delays -> model) ----------

_SINGLE_PUF_A = None  # cache the matrix so we build it only once

def build_single_puf_matrix_A(k: int = K):
    """
    Build the linear map A such that:
        m = A x
    where:
        m: (k+1, ) Arbiter PUF model vector
        x: (4k, ) concatenated delays [a0..a_{k-1}, b0.., c0.., d0..]

    Forward equations (for one PUF, delays a,b,c,d):
        d_i = a_i - b_i
        c_i = c_i - d_i   (note: naming clash, but matches paper's structure)
        alpha_i = (d_i + c_i)/2
        beta_i  = (d_i - c_i)/2

        m[0]   = alpha_0
        m[i]   = alpha_i + beta_{i-1}      for 1 <= i <= k-1
        m[k]   = beta_{k-1}
    """
    global _SINGLE_PUF_A
    if _SINGLE_PUF_A is not None:
        return _SINGLE_PUF_A

    rows = k + 1
    cols = 4 * k
    A = np.zeros((rows, cols), dtype=np.float64)

    # index mapping in x:
    #   a_i -> i
    #   b_i -> k + i
    #   c_i -> 2k + i
    #   d_i -> 3k + i

    # m[0] = alpha_0 = 0.5*(a0 - b0 + c0 - d0)
    i = 0
    A[0, 0 + i]   =  0.5       # a0
    A[0, k + i]   = -0.5       # b0
    A[0, 2*k + i] =  0.5       # c0
    A[0, 3*k + i] = -0.5       # d0

    # m[i] = alpha_i + beta_{i-1}, for 1 <= i <= k-1
    for i in range(1, k):
        # alpha_i = 0.5*(ai - bi + ci - di)
        A[i, 0 + i]   +=  0.5
        A[i, k + i]   += -0.5
        A[i, 2*k + i] +=  0.5
        A[i, 3*k + i] += -0.5

        # beta_{i-1} = 0.5*(a_{i-1} - b_{i-1} - c_{i-1} + d_{i-1})
        j = i - 1
        A[i, 0 + j]   +=  0.5
        A[i, k + j]   += -0.5
        A[i, 2*k + j] += -0.5
        A[i, 3*k + j] +=  0.5

    # last row: m[k] = beta_{k-1}
    i = k - 1
    A[k, 0 + i]   =  0.5
    A[k, k + i]   = -0.5
    A[k, 2*k + i] = -0.5
    A[k, 3*k + i] =  0.5

    _SINGLE_PUF_A = A
    return A


def invert_single_puf_model(m: np.ndarray, k: int = K):
    """
    Given a (k+1,)-dim arbiter PUF model m, solve for non-negative delays
    a,b,c,d of length k using least squares:

        minimize ||A x - m||^2

    then clamp negatives in x to zero and split into the four delay vectors.
    """
    m = np.asarray(m, dtype=np.float64).ravel()
    if m.size != k + 1:
        raise ValueError(f"invert_single_puf_model expects length {k+1}, got {m.size}")

    A = build_single_puf_matrix_A(k)

    # Solve least squares A x ≈ m
    # (we do not enforce non-negativity inside the solver, we clamp afterwards,
    # as allowed in the assignment hint.)
    x, *_ = np.linalg.lstsq(A, m, rcond=None)
    x = x.astype(np.float64)

    # Enforce non-negativity
    x[x < 0.0] = 0.0

    a = x[0      : k     ].copy()
    b = x[k      : 2*k   ].copy()
    c = x[2*k    : 3*k   ].copy()
    d = x[3*k    : 4*k   ].copy()

    return a, b, c, d


# ---------------- decoder ----------------
def my_decode(w: np.ndarray):
    """
    Input:
        w: numpy array length 1089 ((K+1)^2)
    Output:
        tuple of eight numpy arrays (a,b,c,d,p,q,r,s), each shape (K,)
        All outputs are non-negative (zeros allowed).
    """
    expect = (K+1)**2
    w = np.asarray(w, dtype=np.float64).ravel()
    if w.size != expect:
        raise ValueError(f"my_decode expects length {expect}, got {w.size}")

    # 1. reshape and top-rank SVD to de-Kronecker: w ≈ u ⊗ v
    W = w.reshape((K+1, K+1))
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    s0 = S[0] if S.size > 0 else 0.0
    root = np.sqrt(s0) if s0 > 0.0 else 0.0
    u_est = U[:, 0] * root
    v_est = Vt[0, :] * root

    # deterministic sign choice: make last coord of v_est non-negative
    if v_est[-1] < 0.0:
        u_est = -u_est
        v_est = -v_est

    # 2. invert each single-PUF model via linear least squares A x = m
    a, b, c, d = invert_single_puf_model(u_est, k=K)
    p, q, r, s = invert_single_puf_model(v_est, k=K)

    # final clamp for safety
    for arr in (a, b, c, d, p, q, r, s):
        arr[arr < 0.0] = 0.0

    return a, b, c, d, p, q, r, s


# ---------------- reconstruction ----------------
def rebuild_model_from_delays(a, b, c, d):
    alpha = (a - b + c - d) / 2.0
    beta  = (a - b - c + d) / 2.0
    m = np.zeros(K+1, dtype=np.float64)
    m[0] = alpha[0]
    if K > 1:
        m[1:-1] = alpha[1:] + beta[:-1]
    m[-1] = beta[-1]
    return m

def reconstruct_w_from_delays(a,b,c,d,p,q,r,s):
    u = rebuild_model_from_delays(a,b,c,d)
    v = rebuild_model_from_delays(p,q,r,s)
    return np.kron(u, v)

# ---------------- evaluation harness ----------------
def evaluate_models(models, repeats=REPEATS, out_dir=OUTPUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    per_model_summary = []
    all_delays = []
    for i, w in enumerate(models):
        # warm-up
        _ = my_decode(w)
        times = []
        errors = []
        rep_delays = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            a,b,c,d,p,q,r,s = my_decode(w)
            t1 = time.perf_counter()
            elapsed = t1 - t0
            # clamp negatives (mimic grader)
            for arr in (a,b,c,d,p,q,r,s):
                arr[arr < 0.0] = 0.0
            w_hat = reconstruct_w_from_delays(a,b,c,d,p,q,r,s)
            err = float(np.linalg.norm(w - w_hat))
            times.append(elapsed)
            errors.append(err)
            if rep_delays is None:
                rep_delays = np.concatenate([a,b,c,d,p,q,r,s])
        mean_time = float(np.mean(times))
        mean_err = float(np.mean(errors))
        per_model_summary.append({
            "model_index": int(i),
            "mean_time_s": mean_time,
            "mean_L2_error": mean_err,
            "times": times,
            "errors": errors
        })
        all_delays.append(rep_delays)
        # save per-model delays
        md = Path(out_dir) / f"model_{i}"
        md.mkdir(parents=True, exist_ok=True)
        a0 = rep_delays[0:K]; b0 = rep_delays[K:2*K]; c0 = rep_delays[2*K:3*K]; d0 = rep_delays[3*K:4*K]
        p0 = rep_delays[4*K:5*K]; q0 = rep_delays[5*K:6*K]; r0 = rep_delays[6*K:7*K]; s0 = rep_delays[7*K:8*K]
        np.savez(md/"delays.npz", a=a0, b=b0, c=c0, d=d0, p=p0, q=q0, r=r0, s=s0)
        np.savetxt(md/"delays_flat.csv", rep_delays[None,:], delimiter=",", fmt="%.12g")
        print(f"[Model {i}] mean_time = {mean_time:.6e} s, mean_L2_error = {mean_err:.6e}")
    # overall summary
    mean_time_all = float(np.mean([rec["mean_time_s"] for rec in per_model_summary]))
    mean_err_all = float(np.mean([rec["mean_L2_error"] for rec in per_model_summary]))
    np.savez(Path(out_dir)/"results_summary.npz",
             per_model_summary=per_model_summary,
             mean_time_all=mean_time_all,
             mean_err_all=mean_err_all,
             all_delays=np.vstack(all_delays) if all_delays else np.empty((0, 8*K)))
    print("\nOverall mean time (s):", mean_time_all)
    print("Overall mean L2 error :", mean_err_all)
    print("Results and per-model delays saved at:", out_dir)
    return per_model_summary, mean_time_all, mean_err_all

# ---------------- main ----------------
def main(argv):
    # Robust argument handling:
    # - ignore Jupyter/Colab auto args like '-f' or kernel.json
    # - if user provided an explicit path that exists and contains valid tokens, use it
    # - otherwise fall back to DEFAULT_INPUT
    args = [a for a in argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    input_path = None
    if len(args) >= 1:
        cand = args[0]
        if os.path.exists(cand):
            try:
                # quick check: does it contain at least one full model?
                _ = load_models_from_file(cand, k=K, verbose=False)
                input_path = cand
            except Exception as e:
                print(f"[main] Ignoring provided file '{cand}': {e}")
    if input_path is None:
        # fallback to the known uploaded sample file path
        input_path = DEFAULT_INPUT
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"No valid input found. Tried CLI args and fallback '{DEFAULT_INPUT}'")

    print("[main] Using input file:", input_path)
    models = load_models_from_file(input_path, k=K, verbose=True)
    evaluate_models(models, repeats=REPEATS, out_dir=OUTPUT_DIR)

if __name__ == "__main__":
    main(sys.argv)


[main] Using input file: //content/secret_mod.txt
[loader] extracted 10890 numeric tokens from '//content/secret_mod.txt'
[loader] loaded 10 model(s), each length 1089.
[Model 0] mean_time = 2.516572e-03 s, mean_L2_error = 9.224353e-01
[Model 1] mean_time = 3.690346e-03 s, mean_L2_error = 1.156101e+00
[Model 2] mean_time = 5.997918e-03 s, mean_L2_error = 1.130388e+00
[Model 3] mean_time = 6.651817e-03 s, mean_L2_error = 8.635494e-01
[Model 4] mean_time = 6.118893e-03 s, mean_L2_error = 1.149955e+00
[Model 5] mean_time = 1.595005e-03 s, mean_L2_error = 9.223897e-01
[Model 6] mean_time = 1.468543e-03 s, mean_L2_error = 9.913617e-01
[Model 7] mean_time = 1.316615e-03 s, mean_L2_error = 9.791280e-01
[Model 8] mean_time = 1.603258e-03 s, mean_L2_error = 7.755569e-01
[Model 9] mean_time = 1.373554e-03 s, mean_L2_error = 9.501550e-01

Overall mean time (s): 0.0032332523400003766
Overall mean L2 error : 0.9841019702629559
Results and per-model delays saved at: /mnt/data/puf_recovery_output


In [ ]:
#!/usr/bin/env python3
"""
Submit file for Problem 1.2 – XOR Arbiter PUF inversion.

Core idea:
- De-Kronecker w (1089-dim) into u, v (each 33-dim) via a custom rank-1
  factorization based on an anchor element of the 33x33 matrix.
- Invert each 1-PUF model vector (u, v) into non-negative delays (a,b,c,d),
  (p,q,r,s) using an algebraic construction that enforces non-negativity
  via delay-shift invariances.

Main function exposed to the grader:
    my_decode(model)
"""

import numpy as np

# ================== Helper: 1-PUF model -> model (forward) ==================

def _delays_to_model_one_puf(a, b, c, d):
    """
    Given 4 arrays of length k = 32 (delays a_i, b_i, c_i, d_i),
    compute the corresponding arbiter PUF linear model m \in R^{k+1}
    using the formulas from the assignment.

    This is mainly for internal checking / debugging; not used by the grader.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)
    d = np.asarray(d, dtype=float)

    k = a.shape[0]
    assert b.shape[0] == k and c.shape[0] == k and d.shape[0] == k

    alpha = (a - b + c - d) / 2.0
    beta  = (a - b - c + d) / 2.0

    m = np.zeros(k + 1, dtype=float)
    m[0] = alpha[0]
    for i in range(1, k):
        m[i] = alpha[i] + beta[i - 1]
    m[k] = beta[k - 1]
    return m


# ================== Helper: 1-PUF model -> delays (inverse) ==================

def _model_to_delays_one_puf(m):
    """
    Invert a single arbiter PUF model vector m (length 33)
    to one valid set of non-negative delays (a, b, c, d),
    each of length 32.

    Strategy (fixed underdetermined solution):

    Model equations for one PUF (k = 32):
        m[0]     = alpha[0]
        m[i]     = alpha[i] + beta[i-1],   for 1 <= i <= 31
        m[32]    = beta[31]

    We choose:
        alpha[0]   = m[0]
        alpha[i]   = m[i]      for 1 <= i <= 31
        beta[0..30] = 0
        beta[31]    = m[32]

    Then for each stage i:
        X_i = a_i - b_i = alpha_i + beta_i
        Y_i = c_i - d_i = alpha_i - beta_i

    We first set:
        a_i = X_i,  b_i = 0
        c_i = Y_i,  d_i = 0

    Then we exploit invariance under adding a constant to both delays in a pair:
        (a_i, b_i) -> (a_i + t, b_i + t)
        (c_i, d_i) -> (c_i + s, d_i + s)
    choosing t, s so all final delays are non-negative.
    """
    m = np.asarray(m, dtype=float).flatten()
    k_plus_1 = m.shape[0]
    if k_plus_1 != 33:
        raise ValueError(f"Expected model of length 33 for one PUF, got {k_plus_1}")
    k = 32

    alpha = np.zeros(k, dtype=float)
    beta  = np.zeros(k, dtype=float)

    # Fixed choice that reproduces m exactly
    alpha[0]     = m[0]
    alpha[1:k]   = m[1:k]   # m[1]..m[31]
    beta[:]      = 0.0
    beta[k - 1]  = m[k]     # m[32]

    # Now build delays from (alpha, beta)
    a = np.zeros(k, dtype=float)
    b = np.zeros(k, dtype=float)
    c = np.zeros(k, dtype=float)
    d = np.zeros(k, dtype=float)

    for i in range(k):
        X = alpha[i] + beta[i]   # a_i - b_i
        Y = alpha[i] - beta[i]   # c_i - d_i

        # Initial (possibly negative) choice
        a_tmp = X
        c_tmp = Y
        b_tmp = 0.0
        d_tmp = 0.0

        # Shift (a_i, b_i) so both are non-negative
        t = max(0.0, -a_tmp)
        a_i = a_tmp + t
        b_i = b_tmp + t

        # Shift (c_i, d_i) so both are non-negative
        s = max(0.0, -c_tmp)
        c_i = c_tmp + s
        d_i = d_tmp + s

        # Clamp tiny negative noise caused by floating point
        if a_i < 0.0: a_i = 0.0
        if b_i < 0.0: b_i = 0.0
        if c_i < 0.0: c_i = 0.0
        if d_i < 0.0: d_i = 0.0

        a[i] = a_i
        b[i] = b_i
        c[i] = c_i
        d[i] = d_i

    return a, b, c, d


# ================== Helper: de-Kronecker (w -> u, v) ==================

def _recover_uv_from_w(w):
    """
    Given w \in R^{1089}, reshape to 33x33 matrix W and
    factor it as outer product u v^T, where u, v \in R^{33}.

    If W is (numerically) all zeros, return u = v = 0.

    Method:
      - Reshape w -> W (33x33) in row-major order.
      - Find (i0, j0) where |W[i0, j0]| is maximum (anchor).
      - Set:
          u[i] = W[i, j0]
          v[j] = W[i0, j] / W[i0, j0]
        This satisfies W = u v^T exactly when W is rank 1.
    """
    w = np.asarray(w, dtype=float).flatten()
    n = w.shape[0]
    if n != 33 * 33:
        raise ValueError(f"Expected w of length 1089, got {n}")

    k1 = 33
    W = w.reshape((k1, k1))

    # Check for (numerically) zero matrix
    max_abs = np.max(np.abs(W))
    if max_abs < 1e-12:
        # Degenerate case: everything is (near) zero
        u = np.zeros(k1, dtype=float)
        v = np.zeros(k1, dtype=float)
        return u, v

    # Anchor at the largest magnitude entry
    idx_flat = np.argmax(np.abs(W))
    i0, j0 = divmod(int(idx_flat), k1)
    anchor = W[i0, j0]

    # Just in case, avoid division by something very tiny
    if abs(anchor) < 1e-15:
        anchor = np.sign(anchor) * 1e-15 if anchor != 0.0 else 1e-15

    u = W[:, j0].copy()
    v = W[i0, :] / anchor

    return u, v


# ================== Public API for Problem 1.2 ==================

def my_decode(model):
    """
    Main function required by the assignment for Problem 1.2.

    Input:
        model : 1D array-like of length 1089
                Linear model w of a 32-bit XOR arbiter PUF.

    Output:
        eight 1D numpy arrays of length 32:
            a, b, c, d for the first PUF,
            p, q, r, s for the second PUF.
        All entries are guaranteed to be non-negative (zeros allowed).
    """
    w = np.asarray(model, dtype=float).flatten()
    if w.shape[0] != 33 * 33:
        raise ValueError(f"my_decode expects a 1089-d model, got {w.shape[0]}")

    # 1. De-Kronecker: w -> u, v
    u, v = _recover_uv_from_w(w)

    # 2. Invert each 1-PUF model to delays
    a, b, c, d = _model_to_delays_one_puf(u)
    p, q, r, s = _model_to_delays_one_puf(v)

    # Final safety: clamp any tiny negative values to zero
    def _clamp_nonnegative(x):
        x = np.asarray(x, dtype=float)
        x[x < 0.0] = 0.0
        return x

    a = _clamp_nonnegative(a)
    b = _clamp_nonnegative(b)
    c = _clamp_nonnegative(c)
    d = _clamp_nonnegative(d)
    p = _clamp_nonnegative(p)
    q = _clamp_nonnegative(q)
    r = _clamp_nonnegative(r)
    s = _clamp_nonnegative(s)

    return a, b, c, d, p, q, r, s


# ================== Optional sanity check (can be removed for submission) ==================

if __name__ == "__main__":
    # Example sanity check using a file "public_mod.txt"
    # Format: each line = one 1089-d model (space-separated floats)
    try:
        models = []
        with open("/content/secret_mod.txt", "r") as f:
            for line in f:
                vals = list(map(float, line.strip().split()))
                if len(vals) == 1089:
                    models.append(np.array(vals, dtype=float))

        print(f"Loaded {len(models)} models from public_mod.txt")

        for idx, w in enumerate(models):
            a, b, c, d, p, q, r, s = my_decode(w)

            # Reconstruct model to check error
            u = _delays_to_model_one_puf(a, b, c, d)
            v = _delays_to_model_one_puf(p, q, r, s)
            w_recon = np.kron(u, v)

            err = np.linalg.norm(w - w_recon)
            print(f"Model {idx}: reconstruction error = {err:.3e}")

    except FileNotFoundError:
        print("public_mod.txt not found. Skipping sanity check.")


Loaded 10 models from public_mod.txt
Model 0: reconstruction error = 1.090e-16
Model 1: reconstruction error = 1.246e-16
Model 2: reconstruction error = 1.421e-16
Model 3: reconstruction error = 1.121e-16
Model 4: reconstruction error = 1.470e-16
Model 5: reconstruction error = 1.097e-16
Model 6: reconstruction error = 1.317e-16
Model 7: reconstruction error = 1.169e-16
Model 8: reconstruction error = 9.880e-17
Model 9: reconstruction error = 1.279e-16


<>:23: SyntaxWarning: invalid escape sequence '\i'
<>:140: SyntaxWarning: invalid escape sequence '\i'
<>:23: SyntaxWarning: invalid escape sequence '\i'
<>:140: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipython-input-1546260861.py:23: SyntaxWarning: invalid escape sequence '\i'
  compute the corresponding arbiter PUF linear model m \in R^{k+1}
/tmp/ipython-input-1546260861.py:140: SyntaxWarning: invalid escape sequence '\i'
  Given w \in R^{1089}, reshape to 33x33 matrix W and


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

# Create a python file (example content)
python_code = """print("Hello, IIT Kanpur!")"""

with open("/content/submit.py", "w") as f:
    f.write(python_code)

# Create password protected zip
password = "PSSVVY123".encode('utf-8')

with ZipFile("assignment.zip", "w", ZIP_DEFLATED) as zipf:
    zipf.setpassword(password)
    zipf.write("/content/submit.py")

print("Password-protected zip created successfully: assignment.zip")


Password-protected zip created successfully: assignment.zip
